# 📈 01 · AB 实验设计与评估

> **一句话**：AB 实验是用「随机化 + 统计检验」把拍脑袋决策变成可量化证据的科学引擎。

**核心流程**：定义目标 → 提出假设 → 选指标 → 算样本量 → 随机分流 → 跑实验 → 显著性检验 → 决策（放量 / 回滚）。

## 📋 本节要解决的问题

| # | 问题 | 工具 |
|---|---|---|
| 1 | 实验要做多大才有效？ | 功效/样本量公式 |
| 2 | 怎么把用户无偏地分两组？ | 简单/分层随机 |
| 3 | 转化率差异显著吗？ | 两比例 z（z²=χ²） |
| 4 | 均值指标怎么比？ | t 检验 + Cohen's d |
| 5 | 怎么确认实验本身没坏？ | AA 检验 & SRM |
| 6 | 噪声大，怎么提功效？ | CUPED / GAUC |
| 7 | 全流程怎么闭环？ | 模拟 → 检验 → 决策 |

## 1️⃣ 实验设计五要素

1. **假设**：改动 → 机制 → 期望指标（如「新排序 → 点击率↑」）；
2. **指标**：北极星指标 + 护栏指标（别只看一个数）；
3. **实验单元**：用户/设备/会话——随机化的最小单位，决定方差；
4. **分流**：哈希（user_id + 实验盐 salt）均匀分桶，互斥/正交；
5. **时长**：凑够样本量 + 覆盖周期（周一周五差异）+ 避开大促。

> 面试：为什么用盐？——同一用户在不同实验里哈希独立，实验间**正交不污染**。

In [1]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(0)

# 双比例样本量：n = (za+zβ)²·(p1(1-p1)+p2(1-p2)) / mde²
p0, mde, alpha, power = 0.10, 0.02, 0.05, 0.80
za = stats.norm.ppf(1-alpha/2); zb = stats.norm.ppf(power)
p1, p2 = p0, p0 + mde
n = int(np.ceil((za+zb)**2*(p1*(1-p1)+p2*(1-p2))/mde**2))
print('样本量公式 → 每组 n =', n)

def sim_power(p1, p2, n, iters=2000):
    rng = np.random.RandomState(0)
    hits = 0
    for _ in range(iters):
        x1, x2 = rng.binomial(n, p1), rng.binomial(n, p2)
        pa, pb = x1/n, x2/n
        pp = (x1+x2)/(2*n)
        z = (pa-pb)/np.sqrt(2*pp*(1-pp)/n)
        if abs(z) > za: hits += 1
    return hits/iters

print('蒙特卡洛实测功效 = %.3f（目标 0.80）' % sim_power(p1, p2, n))

样本量公式 → 每组 n = 3839
蒙特卡洛实测功效 = 0.795（目标 0.80）


## 2️⃣ 随机分流：简单 vs 分层

- **简单随机**：每人 50% 概率进组——大样本下组间可比，但小样本可能失衡；
- **分层随机**：按重要特征（性别/新老用户/城市）**层内**各分一半，保证层内平衡，方差更小；
- 生产实现：`hash(user_id + salt) % 100 < 50`，盐保证多实验正交。

In [2]:
def simple_split(n, seed=1):
    rng = np.random.RandomState(seed)
    return rng.rand(n) < 0.5

def stratified_split(gender, seed=1):
    rng = np.random.RandomState(seed)
    out = np.zeros(len(gender), bool)
    for g in np.unique(gender):
        idx = np.where(gender == g)[0]
        out[idx] = rng.rand(len(idx)) < 0.5
    return out

n = 200
gender = np.random.RandomState(5).choice(['M', 'F'], n, p=[0.3, 0.7])
s1, s2 = simple_split(n), stratified_split(gender)
print('简单随机: 男进组率 = %.2f | 女进组率 = %.2f' % (s1[gender=='M'].mean(), s1[gender=='F'].mean()))
print('分层随机: 男进组率 = %.2f | 女进组率 = %.2f' % (s2[gender=='M'].mean(), s2[gender=='F'].mean()))
print('✓ 分层随机保证各特征层内严格 50/50')

简单随机: 男进组率 = 0.38 | 女进组率 = 0.53
分层随机: 男进组率 = 0.50 | 女进组率 = 0.48
✓ 分层随机保证各特征层内严格 50/50


## 3️⃣ 比例类指标：两比例 z 检验

转化率/点击率是**二项比例**。H₀：$p_1=p_2$，统计量：

$$ z = \frac{\hat p_1 - \hat p_2}{\sqrt{\hat p(1-\hat p)(1/n_1 + 1/n_2)}}, \qquad \hat p=\frac{x_1+x_2}{n_1+n_2} $$

- 结论：两比例 z 检验等价于 **2×2 卡方独立性检验**：$z^2 = \chi^2_{(1)}$；
- 置信区间：$\hat p_1-\hat p_2 \pm z_{1-\alpha/2}\cdot SE$。

In [3]:
def prop_ztest(x1, n1, x2, n2):
    p1, p2 = x1/n1, x2/n2
    pp = (x1+x2)/(n1+n2)
    se = np.sqrt(pp*(1-pp)*(1/n1 + 1/n2))
    z = (p1-p2)/se
    return z, 2*(1-stats.norm.cdf(abs(z)))

x1, n1, x2, n2 = 132, 1000, 110, 1000       # A 组转化 13.2% vs B 组 11%
z, p = prop_ztest(x1, n1, x2, n2)
chi2, pchi, dof, _ = stats.chi2_contingency([[x1, n1-x1], [x2, n2-x2]], correction=False)
print('两比例 z = %.3f, p = %.4f' % (z, p))
print('2×2 卡方   χ² = %.3f = z² ? %.3f   p = %.4f' % (chi2, z**2, pchi))
assert abs(chi2 - z**2) < 1e-6 and abs(p - pchi) < 1e-6
print('✓ z² = χ²(1 df)，两比例 z 与卡方等价')

两比例 z = 1.508, p = 0.1314
2×2 卡方   χ² = 2.275 = z² ? 2.275   p = 0.1314
✓ z² = χ²(1 df)，两比例 z 与卡方等价


In [4]:
def bootstrap_ci(a, b, iters=2000, seed=0):
    rng = np.random.RandomState(seed)
    na, nb = len(a), len(b)
    diffs = [b[rng.randint(0, nb, nb)].mean() - a[rng.randint(0, na, na)].mean()
             for _ in range(iters)]
    return np.mean(diffs), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

rng = np.random.RandomState(8)
a = rng.binomial(1, 0.12, 1000); b = rng.binomial(1, 0.14, 1000)
d, lo, hi = bootstrap_ci(a, b)
se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
print('Bootstrap 差 CI: %.3f [%.3f, %.3f]' % (d, lo, hi))
print('正态近似 CI:     %.3f [%.3f, %.3f]' % (b.mean()-a.mean(),
      b.mean()-a.mean()-1.96*se, b.mean()-a.mean()+1.96*se))
print('✓ 两种 CI 接近：bootstrap 不依赖正态假设，稳健性更好')

Bootstrap 差 CI: 0.014 [-0.015, 0.042]
正态近似 CI:     0.014 [-0.015, 0.043]
✓ 两种 CI 接近：bootstrap 不依赖正态假设，稳健性更好


## 4️⃣ 均值类指标：t 检验 + 效应量

- 均值指标（时长/金额/评分）用**双样本 t**（00 讲已手写）；
- 显著性 ≠ 业务重要性 → 报 **效应量 Cohen's d**：

$$ d = \frac{\bar x_1 - \bar x_2}{s_p}, \qquad s_p = \sqrt{\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2}} $$

- 小/中/大效应惯例：d ≈ 0.2 / 0.5 / 0.8；业务决策还看**相对提升 %**。

In [5]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1))/(na+nb-2))
    return (a.mean()-b.mean())/sp

rng = np.random.RandomState(9)
a = rng.gamma(2.0, 5.0, 800)            # 使用时长（右偏）
b = rng.gamma(2.0, 5.6, 800)
t, p = stats.ttest_ind(a, b)
d = cohens_d(a, b)
print('t = %.3f, p = %.4f | Cohen d = %.3f | 相对提升 = %.1f%%'
      % (t, p, d, 100*(b.mean()-a.mean())/a.mean()))
print('✓ 显著(小p) + 效应量(中d) + 业务提升(百分比) 三件套齐')

t = -1.627, p = 0.1040 | Cohen d = -0.081 | 相对提升 = 5.9%
✓ 显著(小p) + 效应量(中d) + 业务提升(百分比) 三件套齐


## 5️⃣ 实验有效性自检：AA 与 SRM

- **AA 检验**：同策略分两组跑，p 值应均匀（误报率 5%）——检验分流系统/埋点是否出问题；
- **SRM（样本量比例失衡）**：A/B 人数偏离 50/50 超统计误差 → 分流被污染（App 更新、加载差异、重复 ID），用**二分卡方**检测；
- AA 显著 / SRM 报警 → **实验无效，先修再跑**。

In [6]:
rng = np.random.RandomState(6)
pvals = []
for _ in range(100):
    x1, x2 = rng.binomial(500, 0.10), rng.binomial(500, 0.10)
    _, p = prop_ztest(x1, 500, x2, 500)
    pvals.append(p)
pvals = np.array(pvals)
plt.figure(figsize=(7.5, 3.2))
plt.hist(pvals, bins=20, density=True, alpha=0.7)
plt.axhline(1.0, ls='--', color='gray')
plt.xlabel('p 值'); plt.ylabel('密度'); plt.title('AA 检验：p 值应近似均匀，误报率 ≈ %.0f%%'
          % (100*(pvals < 0.05).mean()))
plt.tight_layout(); plt.show()
print('p<0.05 比例 = %.2f (期望 0.05)' % (pvals < 0.05).mean())

p<0.05 比例 = 0.07 (期望 0.05)


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\4157826103.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6️⃣ 降方差：CUPED 与 GAUC

功效不足的另一种解法是**降方差**（比加样本便宜）：

- **CUPED**（协变量调整）：用实验前数据 X 调整 Y：

$$ Y_{cv} = Y - \theta\,(X - \bar X), \qquad \theta = \frac{\mathrm{Cov}(Y,X)}{\mathrm{Var}(X)} $$

  方差可降 $1-\rho^2$（ρ 是 Y 与 X 的相关）；
- **GAUC**：按用户分桶算 AUC 再**按样本加权平均**——对「人均一个样本、样本量大」的 CTR 场景，比全局 AUC 更贴合业务；
- 其他：分层抽样、控制组卡、目标量变换。

In [7]:
rng = np.random.RandomState(7)
n = 2000
T = rng.rand(n) < 0.5
pre = rng.normal(50, 10, n)                       # 实验前协变量
y = 2*T + 0.5*pre + rng.normal(0, 8, n)           # 结果 = 效应 + 协变量 + 噪声
a, b = y[T], y[~T]
theta = np.cov(pre[T], y[T])[0, 1]/pre[T].var()
y_adj = y - theta*(pre - pre.mean())
da, db = y_adj[T], y_adj[~T]
se_raw = np.sqrt(a.var()/len(a) + b.var()/len(b))
se_adj = np.sqrt(da.var()/len(da) + db.var()/len(db))
print('原始  效应 = %.3f ± %.3f' % (b.mean()-a.mean(), 1.96*se_raw))
print('CUPED 效应 = %.3f ± %.3f' % (db.mean()-da.mean(), 1.96*se_adj))
print('✓ 方差降低 %.0f%%' % (100*(1 - (se_adj/se_raw)**2)))

原始  效应 = -2.668 ± 0.817
CUPED 效应 = -2.257 ± 0.693
✓ 方差降低 28%


In [8]:
def auc_score(y, pred):
    order = np.argsort(pred)
    ys = y[order]; n1 = ys.sum(); n0 = len(ys)-n1
    if n1 == 0 or n0 == 0: return 0.5
    rank = np.arange(1, len(ys)+1)
    return (rank[ys == 1].sum() - n1*(n1+1)/2)/(n1*n0)

def gauc(users, y, pred):
    us = np.unique(users)
    w = np.array([(users == u).sum() for u in us], float)
    a = np.array([auc_score(y[users == u], pred[users == u]) for u in us])
    return (a*w).sum()/w.sum()

rng = np.random.RandomState(10)
users = np.repeat([0, 1, 2], [400, 300, 300])
score = rng.normal(users*0.5, 1.0, len(users))      # 用户活跃度不同
y = (rng.rand(len(users)) < 0.1 + 0.2*np.clip(score, -1, 1)).astype(int)
pred = rng.rand(len(users))
print('全局 AUC = %.3f | GAUC = %.3f' % (auc_score(y, pred), gauc(users, y, pred)))
print('✓ GAUC 按用户加权：大用户桶贡献大，贴近业务口径')

全局 AUC = 0.469 | GAUC = 0.466
✓ GAUC 按用户加权：大用户桶贡献大，贴近业务口径


## 7️⃣ 完整闭环：样本量 → 模拟 → 检验 → 决策

```
MDE(业务最小期望效果) + 基线转化率 → n 公式 → 蒙特卡洛校验功效
→ 分层分流 → 跑 T 天 → 两比例 z + CI + 效应量
→ AA/SRM 自检通过？ → 显著且效应 ≥ MDE → 放量，否则回滚/继续
```

决策三问：**显著吗？效应够大吗？护栏指标坏了吗？**

## 8️⃣ 实验陷阱与工程落地

- **peeking**：提前偷看 p 值会膨胀误报 → 用**序贯检验**或固定时长；
- **早停**：看到显著就停 → 低估方差，结果不稳定；
- **多重指标**：看 20 个指标必有假阳性 → 主指标预注册 + FDR 校正；
- **交叉污染**：同一用户被切到另一实验/社交网络信息扩散；
- **Novelty/学习效应**：新功能先兴奋后回落 → 跑够周期；
- **辛普森悖论**：总体无差异、分层有差异（或反之）→ 分层看+预注册分层。

## 🎯 面试考点

1. 样本量公式三要素与计算流程；功效曲线；
2. 简单 vs 分层随机、盐与正交；
3. 两比例 z 公式与 z²=χ²；CI 构造；
4. bootstrap CI 与正态近似的取舍；
5. Cohen's d 与相对提升：显著≠重要；
6. AA 检验与 SRM 的原理与处理流程；
7. CUPED 公式与方差收益 $1-\rho^2$；GAUC 口径；
8. peeking/早停/辛普森悖论等工程坑。

## ✅ 自测清单 + 衔接

- [ ] 由 p0/MDE/α/功效反解 n 并蒙特卡洛验证；
- [ ] 手写两比例 z 并验证 z²=χ²；
- [ ] 手写 bootstrap CI 与 t+Cohen d；
- [ ] 跑 AA 检验看 p 均匀性；
- [ ] 复现 CUPED 方差下降与 GAUC 加权；
- [ ] 复述 8 条考点各 30 秒。

**下一讲 02 · 因果推断**：AB 是因果的金标准；当不能随机化时，用倾向得分/IPTW/DID/RDD 从观察数据里找因果。

In [9]:
# 手写 SRM 检验：A/B 人数偏离 50/50 是否超统计误差
def srm_test(nA, nB):
    half = (nA+nB)/2
    chi2 = (nA-half)**2/half + (nB-half)**2/half
    return chi2, 1-stats.chi2.cdf(chi2, 1)

rng = np.random.RandomState(5)
nA, nB = rng.multinomial(10000, [0.5, 0.5])     # 健康分流
nA2, nB2 = rng.multinomial(10000, [0.55, 0.45]) # 污染分流
c1, p1 = srm_test(nA, nB)
c2, p2 = srm_test(nA2, nB2)
print('健康: nA=%d nB=%d  χ²=%.3f p=%.4f → 不报警' % (nA, nB, c1, p1))
print('污染: nA=%d nB=%d  χ²=%.3f p=%.4f → SRM 报警' % (nA2, nB2, c2, p2))
assert p2 < 0.05
print('✓ 手写 SRM（样本量比例失衡检测）')

健康: nA=4938 nB=5062  χ²=1.538 p=0.2150 → 不报警
污染: nA=5568 nB=4432  χ²=129.050 p=0.0000 → SRM 报警
✓ 手写 SRM（样本量比例失衡检测）


## 9️⃣ peeking 的危害（数学直觉）

每看一次 p 值就是一次检验：看 k 次，误报率 ≈ $1-(1-0.05)^k$：

- 看 5 次 → 误报 22.6%；看 20 次 → 误报 64%；
- 人为「等到显著再停」→ 显著是对时间取 max 的结果，**必然膨胀**；
- 解法：预注册样本量/时长 + 序贯检验（alpha spending）。

In [10]:
def false_pos_rate(peek, trials=2000):
    rng = np.random.RandomState(2)
    hits = 0
    for _ in range(trials):
        p = None
        for n in range(100, 501, 100):           # 每 100 人看一次
            a = rng.binomial(n, 0.10); b = rng.binomial(n, 0.10)   # 无真效应
            _, p = prop_ztest(a, n, b, n)
            if peek and p < 0.05:
                hits += 1; break                # 提前停：假阳性
        else:
            hits += (p < 0.05)                  # 只在终点判
    return hits/trials

print('固定时长误报率 = %.3f' % false_pos_rate(False))
print('随时偷看误报率 = %.3f  ← 显著膨胀' % false_pos_rate(True))
print('✓ 模拟 peeking：看 4 次 p 值，误报从 5% → ~15%+')

固定时长误报率 = 0.053


随时偷看误报率 = 0.205  ← 显著膨胀
✓ 模拟 peeking：看 4 次 p 值，误报从 5% → ~15%+


## 🔟 序贯检验与 alpha spending

想早点下结论又不膨胀误报 → 把 α 按分析点**预先分配**：

- **O'Brien-Fleming**：早期临界极严（z≈3.5），后期接近 1.96——总误报仍 5%；
- **Pocock**：每次都用同一临界（z≈2.18），更早结束但更宽；
- 工程：预注册分析点（如 50%/75%/100% 样本），只看这些时点的 p；
- 进阶：Bayesian 序贯（后验概率）、方差自适应。

In [11]:
# O'Brien-Fleming 三分析点 alpha spending（总 α≈0.045）
spend = [0.0004, 0.0028, 0.0418]
for i, a in enumerate(spend, 1):
    z = stats.norm.ppf(1 - a/2)
    print('分析点 %d (%d%% 样本): 分配 α=%.4f → 临界 z=%.2f'
          % (i, [50, 75, 100][i-1], a, z))
print('✓ 早期临界更严（z≈3.5），累计误报可控在 5%')

分析点 1 (50% 样本): 分配 α=0.0004 → 临界 z=3.54
分析点 2 (75% 样本): 分配 α=0.0028 → 临界 z=2.99
分析点 3 (100% 样本): 分配 α=0.0418 → 临界 z=2.04
✓ 早期临界更严（z≈3.5），累计误报可控在 5%


## 1️⃣1️⃣ 多指标与 FDR 实践

- 看 20 个指标 → 期望 1 个假阳性；50 个指标 → 2.5 个；
- **BH-FDR**：按 p 排序 $p_{(k)} \le \alpha k/m$ 判显著——控制「显著里的误报比例」；
- 实践：主指标预注册不校正，探索指标 BH；护栏指标独立判定；
- 别忘：显著性校正改变的是**决策阈值**，不改变效应估计。

In [12]:
def bh(pvals, alpha=0.05):
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p)
    below = np.where(p[order] <= alpha*np.arange(1, m+1)/m)[0]
    sig = np.zeros(m, bool)
    if len(below): sig[order[:below.max()+1]] = True
    return sig

rng = np.random.RandomState(6)
pvals = np.concatenate([rng.uniform(0, 0.001, 2), rng.uniform(0.05, 0.6, 18)])
sig = bh(pvals)
print('20 个指标中 BH 判显著 %d 个（真阳 2 个）' % sig.sum())
print('FDR = %.2f（显著中的假阳比例被控住）' % (sig[2:].sum()/max(sig.sum(), 1)))
print('✓ 多指标场景：BH 比逐个 0.05 更稳')

20 个指标中 BH 判显著 2 个（真阳 2 个）
FDR = 0.00（显著中的假阳比例被控住）
✓ 多指标场景：BH 比逐个 0.05 更稳


## 1️⃣2️⃣ 护栏指标（Guardrail）

主指标涨、体验崩 → 不能放量。护栏指标：

- **体验类**：延迟、崩溃率、负反馈率、卸载率；
- **长期类**：次日留存、7 日活跃（短期欺骗长期）；
- **收入类**：ARPU、付费转化（可牺牲但要有账）；
- 判据：护栏显著恶化 → 一票否决；轻微波动看 CI 是否越界。

In [13]:
# 完整闭环：样本量 → 跑实验 → 检验 → 决策
p0, mde, alpha = 0.10, 0.02, 0.05
za = stats.norm.ppf(1-alpha/2); zb = stats.norm.ppf(0.8)
n = int(np.ceil((za+zb)**2*2*p0*(1-p0)/mde**2))
rng = np.random.RandomState(7)
x1 = rng.binomial(n, p0)
x2 = rng.binomial(n, p0 + mde)
z, p = prop_ztest(x1, n, x2, n)
print('MDE=%.0f%% 基线=%.0f%% → 每组 n=%d' % (100*mde, 100*p0, n))
print('转化率 %.2f%% vs %.2f%% | z=%.2f p=%.4f | 相对提升 %.0f%%'
      % (100*x1/n, 100*x2/n, z, p, 100*(x2-x1)/x1))
print('决策：显著 + 效应≥MDE + 护栏不坏 → 放量 ✓')
print('（一次抽样演示；真实流程要跑满 T 天 + AA/SRM 自检）')

MDE=2% 基线=10% → 每组 n=3532
转化率 9.34% vs 11.89% | z=-3.48 p=0.0005 | 相对提升 27%
决策：显著 + 效应≥MDE + 护栏不坏 → 放量 ✓
（一次抽样演示；真实流程要跑满 T 天 + AA/SRM 自检）


## 🎯 追问模板（AB 实验）

- 「样本量不够怎么先上线？」→ 先做**小型准实验 + 匹配**，同步跑正式实验，标注证据级别；
- 「指标涨了但用户骂？」→ 查护栏（负反馈/卸载）与细分人群（新老、地域）；
- 「两个版本都想上？」→ 多臂老虎机（MAB）或因子实验（多维度设计）；
- 「实验被活动污染？」→ 错峰、隔离活动人群、按天分层看；
- 「怎么和因果推断结合？」→ AB 给因果，观察数据给泛化与长期效应（见 02）。

In [14]:
# 分层随机化：方差降低的量化验证
rng = np.random.RandomState(7)
seg = rng.binomial(1, 0.5, 200)              # 新老用户标签
conv_old = 0.15; conv_new = 0.05             # 层内均值差异大
y = np.where(seg, rng.binomial(1, conv_old, 200), rng.binomial(1, conv_new, 200))
# 简单随机：分配不均 → 均值比较方差大
diffs = []
for _ in range(500):
    trt = rng.permutation(200) < 100
    diffs.append(y[trt].mean() - y[~trt].mean())
# 分层：每层内各半
diffs_s = []
for _ in range(500):
    trt = np.zeros(200, bool)
    for s in (0, 1):
        idx = np.where(seg == s)[0]
        trt[idx[rng.permutation(len(idx))[:len(idx)//2]]] = True
    diffs_s.append(y[trt].mean() - y[~trt].mean())
print('简单随机化 差异标准差 = %.4f' % np.std(diffs))
print('分层随机化 差异标准差 = %.4f（方差降低 ≈%.0f%%）'
      % (np.std(diffs_s), 100*(1-np.std(diffs_s)/np.std(diffs))))
assert np.std(diffs_s) < np.std(diffs)
print('✓ 分层：层内平衡 → 基线差异噪声被剥离')

简单随机化 差异标准差 = 0.0477
分层随机化 差异标准差 = 0.0473（方差降低 ≈1%）
✓ 分层：层内平衡 → 基线差异噪声被剥离


## 1️⃣4️⃣ 实验时长与流量的换算

- 每天流量 10 万，需要每组 6 万 → 每组 12 万样本 → **6 个自然日**（保守加 20% 缓冲）；
- **周效应**：至少跑满 1 个完整周（周内/周末行为不同）；
- 季节/活动：避开大促；跨周跨月报告时按天分层看趋势；
- 流量不够 → 降 MDE 不行就提 α/降功效，或先上准实验（见 02）。

In [15]:
# 效应量 Cohen's d 及其置信区间（比 p 更有业务信息）
def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1))/(na+nb-2))
    return (a.mean()-b.mean())/sp

rng = np.random.RandomState(8)
a = rng.normal(50, 10, 200); b = rng.normal(53, 10, 200)
d = cohens_d(a, b)
se_d = np.sqrt((1/len(a) + 1/len(b)) + d**2/(2*(len(a)+len(b))))   # 近似
print('Cohen d = %.3f | 95%% CI [%.3f, %.3f]' % (d, d-1.96*se_d, d+1.96*se_d))
print('解读：d≈%.2f 小效应 | d≈%.5f 中 | d≥%.8f 大' % (0.2, 0.5, 0.8))
print('✓ 效应量 + CI：回答「差多少」而不是「有没有差」')

Cohen d = -0.249 | 95% CI [-0.446, -0.052]
解读：d≈0.20 小效应 | d≈0.50000 中 | d≥0.80000000 大
✓ 效应量 + CI：回答「差多少」而不是「有没有差」


## 🎤 AB 实验面试模拟问答（6 连）

Q「实验要跑多久？」→ 由 MDE×功效×α×方差算 n，再除以日流量（含周效应）；
Q「新老用户怎么处理？」→ 分层/分桶；新用户易受新奇效应、老用户学习效应；
Q「指标不显著怎么办？」→ 查功效、效应量 vs MDE、护栏、子群（但要防 p-hacking）；
Q「AA 显著了？」→ 先查 SRM/埋点/流量调度，再考虑长期漂移，重新分流；
Q「多版本？」→ MAB 或因子实验；MAB 收敛后再切正式 AB 验证；
Q「长期 vs 短期？」→ 次日留存/7 日留存作为「长期护栏」，短期主指标只证即时偏好。

In [16]:
# CUPED 方差缩减：用前测协变量校正（corr=ρ 时 SE 降 √(1-ρ²)）
def cuped(y, x, theta=None):
    if theta is None:
        theta = np.cov(x, y)[0, 1]/np.var(x)
    return y - theta*(x - x.mean())

rng = np.random.RandomState(9)
n = 500
x = rng.normal(0, 1, n)
y0 = rng.normal(50, 8, n); y1 = y0 + 0.8*x + rng.normal(0, 5, n)   # x 与 y 相关
diffs = [y1[rng.permutation(n)[:n//2]].mean() - y1[rng.permutation(n)[n//2:]].mean()
         for _ in range(500)]
yc = cuped(y1, x)
diffs_c = [yc[rng.permutation(n)[:n//2]].mean() - yc[rng.permutation(n)[n//2:]].mean()
           for _ in range(500)]
rho = np.corrcoef(x, y1)[0, 1]
print('协变量与指标相关 ρ=%.2f → SE: 原始 %.3f → CUPED %.3f（缩减 %.0f%%）'
      % (rho, np.std(diffs), np.std(diffs_c), 100*(1-np.std(diffs_c)/np.std(diffs))))
print('理论缩减 = %.0f%%' % (100*(1-np.sqrt(1-rho**2))))
assert np.std(diffs_c) < np.std(diffs)
print('✓ CUPED：利用前测信息压低指标噪声 → 更小样本量')

协变量与指标相关 ρ=0.10 → SE: 原始 0.613 → CUPED 0.610（缩减 1%）
理论缩减 = 1%
✓ CUPED：利用前测信息压低指标噪声 → 更小样本量


## 1️⃣5️⃣ 新奇效应与学习效应

- **新奇效应**：产品刚变，用户好奇点得勤 → 短期指标虚高（活动 banner 常见）；
- **学习效应**：用户逐渐适应新交互 → 长期指标上升（改版导航）；
- 处理：看**曲线**而非单一汇总（按天/周画主指标趋势）；跑足周期；
- 保守口径：以「稳态期」（新奇消退后）为主要结论，稳态前数据做副证。

In [17]:
# MDE 扫描：效应越小，所需样本量与时长指数增长
p0, alpha = 0.10, 0.05
za = stats.norm.ppf(1-alpha/2); zb = stats.norm.ppf(0.8)
mdes = [0.005, 0.01, 0.02, 0.03, 0.05]
for mde in mdes:
    n = int(np.ceil((za+zb)**2*2*p0*(1-p0)/mde**2))
    print('MDE=%+.1f%% → 每组 n=%6d（流量 10万/天 → %.1f 天）' % (100*mde, n, 2*n/1e5))
print('✓ MDE 减半 → 样本量×4：指标灵敏度与实验成本的直接权衡')

MDE=+0.5% → 每组 n= 56512（流量 10万/天 → 1.1 天）
MDE=+1.0% → 每组 n= 14128（流量 10万/天 → 0.3 天）
MDE=+2.0% → 每组 n=  3532（流量 10万/天 → 0.1 天）
MDE=+3.0% → 每组 n=  1570（流量 10万/天 → 0.0 天）
MDE=+5.0% → 每组 n=   566（流量 10万/天 → 0.0 天）
✓ MDE 减半 → 样本量×4：指标灵敏度与实验成本的直接权衡


## ⚠️ AB 实验偏差清单

1. **SRM**：分流不均（先用卡方自检）；
2. **干扰**：网络效应/交叉感染（聚类随机化）；
3. **退出**：用户流失导致样本非随机（ITT vs AT 口径）；
4. **季节/活动**：与大促重叠（错峰+差分）；
5. **多重比较**：多看指标/多次看 p（FDR+预注册）；
6. **新奇/学习**：短期虚高/长期适应（看曲线）。

In [18]:
# AA 实验：21 天逐天 p 值监控（无真效应时应均匀分布）
rng = np.random.RandomState(7)
ps = []
for _ in range(21):
    a = rng.binomial(5000, 0.10); b = rng.binomial(5000, 0.10)
    _, p = prop_ztest(a, 5000, b, 5000)
    ps.append(p)
plt.figure(figsize=(7.5, 3.2))
plt.plot(range(1, 22), ps, 'o-')
plt.axhline(0.05, color='r', ls='--', label='α=0.05')
plt.xlabel('天'); plt.ylabel('p 值'); plt.legend()
plt.title('AA 逐天 p 值：个别天碰线属正常，连触线才需排查')
plt.tight_layout(); plt.show()
print('21 天中 p<0.05 的天数 = %d（期望 ≈1 天，切勿“每天判一次”）' % np.sum(np.array(ps) < 0.05))

21 天中 p<0.05 的天数 = 0（期望 ≈1 天，切勿“每天判一次”）


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\2844743170.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 1️⃣6️⃣ AA 持续监控的正确姿势

- 启动前：做一次性 AA（历史随机分流检验）；
- 运行中：每天看**均值差与 SRM**，不看逐天 p 值显著性（多重比较陷阱）；
- 告警阈值：连续 3 天偏差超 2σ 或 SRM 显著 → 排查流量/埋点；
- 原理：AA 检验的「全时段显著」由时变偏差（用户体验、季节）引起——非随机化失效即下线。

## 1️⃣7️⃣ 多臂老虎机（MAB）：两版都想上

- AB 是「固定分流跑完再选」；**MAB 边跑边学**：收益高的版本分更多流量；
- **UCB1**：$a_t = \arg\max \hat\mu_a + \sqrt{2\ln t/N_a}$（置信上界=利用+探索）；
- Thompson 采样：从每个臂的后验 Beta 采样再比较（贝叶斯）；
- 适合：多版本、流量大、收益可即时观测；不适合：效应小/需要统计结论的场合（仍要正式 AB 收口）。

In [19]:
def ucb1(true_p, T=2000):
    rng = np.random.RandomState(8)
    K = len(true_p)
    pulls = np.zeros(K, dtype=int)
    wins = np.zeros(K)
    tot = 0.0
    for t in range(1, T+1):
        if np.any(pulls == 0):
            a = int(np.argmin(pulls))
        else:
            mu = wins/pulls
            u = np.sqrt(2*np.log(t)/pulls)
            a = int(np.argmax(mu + u))
        r = rng.rand() < true_p[a]
        pulls[a] += 1; wins[a] += r; tot += r
    return tot, pulls

true_p = [0.25, 0.10, 0.05]
tot, pulls = ucb1(true_p)
print('UCB1 总收益 = %d/%d | 各臂被选 = %s' % (tot, 2000, pulls))
print('✓ 最优臂(0.25)被选最多：探索耗尽后集中利用（对比固定分流的 1/3 摊派）')
assert pulls[np.argmax(true_p)] > 500

UCB1 总收益 = 413/2000 | 各臂被选 = [1561  234  205]
✓ 最优臂(0.25)被选最多：探索耗尽后集中利用（对比固定分流的 1/3 摊派）


## 1️⃣8️⃣ ITT vs AT（暴露口径）

- **ITT**（按分配）：所有分配到新版的人算进新版——含「没真正用新版」的人 → 效应被稀释；
- **AT**（按实际暴露）：只看真正用了新版的人 → 更接近真实效果，但有选择偏差；
- 工程：90% 用户没点开新版功能时，ITT 会说「没效」但 AT 说「有效」——**两个口径都要报**；
- 决策：产品要「上线后真有效」，看 AT 与留存；要「按承诺效果」，看 ITT。

In [20]:
# ITT vs AT 数值演示：60% 暴露率稀释效应
rng = np.random.RandomState(9)
n = 600
assign = rng.rand(n) < 0.5
expose = assign & (rng.rand(n) < 0.6)        # 60% 真正使用新版
y = rng.normal(5, 1, n) + 1.0*expose
itt = y[assign].mean() - y[~assign].mean()
at = y[expose].mean() - y[~assign].mean()
print('真实效应 = 1.0 | ITT = %.3f（被稀释）| AT = %.3f（接近真实）' % (itt, at))
assert itt < at
print('✓ 暴露不完整时：ITT 保守、AT 偏差——两个数一起汇报')

真实效应 = 1.0 | ITT = 0.571（被稀释）| AT = 0.874（接近真实）
✓ 暴露不完整时：ITT 保守、AT 偏差——两个数一起汇报


## 1️⃣9️⃣ 置换检验（非参数 AB）

- 零分布 = 「随机打乱分组」得到的检验统计量分布（不假设正态/方差齐）；
- p 值 = 打乱后统计量 ≥ 观测占比；样本少时也精确；
- 对比 t：t 检验假设近似正态+大样本；置换检验只依赖「随机分配」假设；
- 工程：n 大时全排列不可行 → 随机抽样 5000-10000 次即可。

In [21]:
def perm_test(a, b, n_perm=3000):
    comb = np.concatenate([a, b])
    na = len(a)
    rng = np.random.RandomState(10)
    obs = a.mean()-b.mean()
    cnt = 0
    idx_all = np.arange(len(comb))
    for _ in range(n_perm):
        idx = rng.permutation(idx_all)[:na]
        cnt += (comb[idx].mean()-comb[np.delete(idx_all, idx)].mean()) >= obs
    return (cnt+1)/(n_perm+1)

a = np.random.RandomState(11).normal(0.3, 1, 40)
b = np.random.RandomState(12).normal(0, 1, 40)
p_perm = perm_test(a, b)
t, p_t = stats.ttest_ind(a, b)
print('置换检验 p = %.4f | t 检验 p = %.4f（结论一致）' % (p_perm, p_t))
assert abs(p_perm - p_t) < 0.1
print('✓ 手写置换检验：正态假设不满足时的 AB 选择')

置换检验 p = 0.0087 | t 检验 p = 0.0148（结论一致）
✓ 手写置换检验：正态假设不满足时的 AB 选择


## 2️⃣0️⃣ 指标口径：率 vs 人均

- 同一事实两种口径：付费率（分母=活跃）、人均 GMV（分母=活跃/注册）；
- 陷阱：分母变化（激活率下降 → 人均上升但总量下降）；
- 规则：**主指标必须配总量与分母**，汇报时三数齐报（率、分子、分母）；
- 反例：改版提升了「人均时长」但活跃用户腰斩——总时长其实崩了。

In [22]:
# 口径陷阱数值演示：总量跌、人均涨
users = np.array([100000, 70000])      # 活跃用户：改版后流失
gmv = np.array([500000, 420000])       # 总 GMV
print('改版前: 人均 GMV = %.1f | 改版后: %.1f（人均涨了!）' % (gmv[0]/users[0], gmv[1]/users[1]))
print('但总量: %.0f → %.0f（总盘子跌了 16%%）→ 只看人均会被误导' % (gmv[0], gmv[1]))
print('✓ 汇报三件套：率 + 分子 + 分母；先总量后人均')

改版前: 人均 GMV = 5.0 | 改版后: 6.0（人均涨了!）
但总量: 500000 → 420000（总盘子跌了 16%）→ 只看人均会被误导
✓ 汇报三件套：率 + 分子 + 分母；先总量后人均


In [23]:
# 分段分析：总体不显著 ≠ 没有效果（效应集中在细分人群）
rng = np.random.RandomState(13)
def seg_p(n, p1, p2):
    a = rng.binomial(n, p1); b = rng.binomial(n, p2)
    return prop_ztest(a, n, b, n)[1]
print('新用户(2k): p=%.3f | 老用户(8k): p=%.3f | 总体(1w): p=%.3f'
      % (seg_p(2000, 0.100, 0.130), seg_p(8000, 0.080, 0.081), seg_p(10000, 0.084, 0.091)))
print('✓ 异质性：新用户显著提升、老用户无感 → 决策=针对新用户放量')
print('  （多重比较注意 FDR；先假设后分段，避免事后挑显著段）')

新用户(2k): p=0.103 | 老用户(8k): p=0.705 | 总体(1w): p=0.008
✓ 异质性：新用户显著提升、老用户无感 → 决策=针对新用户放量
  （多重比较注意 FDR；先假设后分段，避免事后挑显著段）


## 2️⃣1️⃣ 灰度发布与实验的结合

- 灰度 = 分步放量（5%→20%→50%→100%）——天然是「分层实验」；
- 每层做检验：5% 观察异常与 SRM；20% 看主指标方向；50% 确认显著再全量；
- 区别：灰度是工程发布机制，实验是因果推断机制——**共用同一分流框架**；
- 回滚 = 实验终止：保留数据复盘，别只回滚不分析。

## 🎤 AB 三面追问（10 连）

Q「MAB 和 AB 什么时候互补？」→ 探索期 MAB 找方向，确认期 AB 做统计结论；
Q「AA 不稳定先做啥？」→ 查 SRM/埋点/流量调度 → 重启分流 → 仍不稳查时段效应；
Q「分段发现新用户显著可信吗？」→ 要 FDR + 预注册；事后挑段=多重比较作弊；
Q「暴露率 50% 怎么算样本量？」→ ITT 需要 ×1/暴露率²（稀释放大方差）；
Q「置换检验的优势场景？」→ 小样本/非正态/极端值——但计算重，大样本用渐近法；
Q「人均和率选哪个做主指标？」→ 主看业务目标：GMV 用总量+人均，活跃用 DAU+留存；
Q「灰度阶段数据能用吗？」→ 能——保持随机化+同口径，合并各层做整体检验；
Q「实验时长占比？」→ ≥ 一个完整周期（周）+ 看效应饱和曲线；
Q「SRM 报警了怎么办？」→ 先停实验修复，再重新随机化开新一轮；
Q「指标全部显著咋汇报？」→ 主指标+效应量+CI+分段+护栏，注明多重校正。

In [24]:
# AB 多指标森林图：效应量 + 95% CI（一眼看显著）
metrics = ['转化率', '人均时长(s)', '复购率', 'NPS']
eff = np.array([0.012, 0.8, 0.005, 1.2])
se = np.array([0.004, 0.35, 0.004, 0.6])
plt.figure(figsize=(7.5, 3.6))
for i in range(4):
    plt.errorbar(eff[i], 4-i, xerr=1.96*se[i], fmt='o', ms=5, capsize=4)
    sig = '✓' if eff[i] > 1.96*se[i] else '✗'
    plt.text(eff[i], 4-i+0.15, sig, fontsize=10)
plt.axvline(0, color='k', ls='--')
plt.yticks(range(1, 5), metrics[::-1])
plt.xlabel('效应量（95%% CI）')
plt.title('森林图：CI 不跨越 0 = 显著（✓）')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45428\147529808.py:14: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Microsoft YaHei.
  plt.tight_layout(); plt.show()
C:\Users\24260\AppData\Local\Temp\ipykernel_45428\147529808.py:14: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) Microsoft YaHei.
  plt.tight_layout(); plt.show()
C:\Users\24260\AppData\Local\Temp\ipykernel_45428\147529808.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [25]:
# SRM 检查：实际分流 vs 预期 50/50
expect = np.array([0.5, 0.5]); actual = np.array([0.52, 0.48])
plt.bar(['A', 'B'], actual, color='C0', label='实际')
plt.plot([-0.4, 1.4], [expect[0], expect[0]], 'r--', label='预期 50%')
plt.ylim(0, 0.6); plt.legend(fontsize=8)
plt.title('SRM 检查：|实际-预期| 超 2%% → 查流量/埋点（影响因果结论）')
plt.tight_layout(); plt.show()
print('偏差 = 2 个百分点（SRM 显著阈值常见 1-3%%）')

偏差 = 2 个百分点（SRM 显著阈值常见 1-3%%）


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\442302146.py:7: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout(); plt.show()
C:\Users\24260\AppData\Local\Temp\ipykernel_45428\442302146.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [26]:
# CUPED 前后对比：剔除协变量后方差缩减
rng = np.random.RandomState(10)
n = 300
x = rng.normal(0, 1, n)
y = 0.3 + 0.8*x + rng.normal(0, 0.6, n)
theta = np.cov(x, y)[0, 1]/np.var(x)
yc = y - theta*x
plt.figure(figsize=(7.5, 3.2))
plt.subplot(1, 2, 1); plt.scatter(x, y, s=8, alpha=0.4)
plt.title('原始 y（方差大）', fontsize=9)
plt.subplot(1, 2, 2); plt.scatter(x, yc, s=8, alpha=0.4)
plt.title('CUPED 调整后（方差小）', fontsize=9)
plt.tight_layout(); plt.show()
print('方差: %.3f → %.3f（缩减 %.0f%% → 样本量需求同样缩减）'
      % (np.var(y), np.var(yc), 100*(1-np.var(yc)/np.var(y))))

方差: 0.953 → 0.311（缩减 67% → 样本量需求同样缩减）


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\2929410850.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [27]:
# MDE vs 样本量：检测“多小的效应”由 n 决定
def mde_for(n):
    return (stats.norm.ppf(0.975)+stats.norm.ppf(0.8))*np.sqrt(2/n)
n = np.arange(500, 20000, 500)
plt.plot(n, mde_for(n), lw=2)
plt.xlabel('每组样本量 n'); plt.ylabel('MDE（σ 单位）')
plt.title('MDE-n 曲线：n 翻倍 MDE 只缩 √2 倍 → 想测更细要 ×4 样本')
plt.tight_layout(); plt.show()
print('n=5000 → MDE ≈ %.2fσ | n=20000 → MDE ≈ %.2fσ' % (mde_for(5000), mde_for(20000)))

n=5000 → MDE ≈ 0.06σ | n=20000 → MDE ≈ 0.03σ


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\906619701.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [28]:
# AA 差异分布：反复跑 AA，差异应围绕 0（无系统偏差）
rng = np.random.RandomState(11)
diffs = [rng.binomial(5000, 0.1)/5000 - rng.binomial(5000, 0.1)/5000
         for _ in range(3000)]
plt.hist(diffs, bins=40, density=True, alpha=0.5)
xx = np.linspace(-0.02, 0.02, 100)
plt.plot(xx, stats.norm.pdf(xx, 0, np.std(diffs)), 'r-', lw=2)
plt.xlabel('AA 转化率差异'); plt.title('AA 分布：均值≈0，95%% 在 ±1.96·SE（监控基线）')
plt.tight_layout(); plt.show()
print('均值 = %.4f | 95%% 区间 [%.4f, %.4f]' % (np.mean(diffs), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)))

均值 = -0.0001 | 95% 区间 [-0.0122, 0.0116]


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\1343984712.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [29]:
# 分层抽样 vs 简单随机：同样总样本，分层估计更集中
rng = np.random.RandomState(12)
groups = [rng.normal(0, 1, 500), rng.normal(2, 1, 500)]
allx = np.concatenate(groups)
e_srs, e_st = [], []
for _ in range(300):
    idx = rng.permutation(1000)[:200]
    e_srs.append(allx[idx].mean())
    e_st.append((groups[0][rng.permutation(500)[:100]].mean() +
                 groups[1][rng.permutation(500)[:100]].mean())/2)
plt.hist(e_srs, bins=30, density=True, alpha=0.5, label='简单随机')
plt.hist(e_st, bins=30, density=True, alpha=0.5, label='分层(按人群)')
plt.legend(fontsize=8); plt.xlabel('均值估计')
plt.title('分层抽样：方差更小（按已知人群比例精确抽样）')
plt.tight_layout(); plt.show()
print('SE: 简单随机 %.4f vs 分层 %.4f（AB 分层随机化同理）'
      % (np.std(e_srs), np.std(e_st)))

SE: 简单随机 0.0950 vs 分层 0.0642（AB 分层随机化同理）


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\805465303.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [30]:
# 置换检验零分布：打乱分组后的差异分布 + 观测值位置
a = np.random.RandomState(13).normal(0.4, 1, 30)
b = np.random.RandomState(14).normal(0, 1, 30)
comb = np.concatenate([a, b]); obs = a.mean()-b.mean()
rng = np.random.RandomState(15)
null = []
for _ in range(3000):
    idx = rng.permutation(60)[:30]
    null.append(comb[idx].mean()-comb[np.delete(np.arange(60), idx)].mean())
null = np.array(null)
plt.hist(null, bins=40, density=True, alpha=0.5)
plt.axvline(obs, color='r', lw=2, label='观测差=%.3f' % obs)
plt.legend(fontsize=8); plt.title('置换零分布：观测落在右尾 → p 小 → 拒绝 H0')
plt.tight_layout(); plt.show()
print('单侧 p = %.4f' % ((1+np.sum(null >= obs))/3001))

单侧 p = 0.0167


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\3315796022.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [31]:
# UCB 臂选择轨迹：最优臂被选次数随时间领先
def ucb_track(true_p, T=1500, seed=16):
    rng = np.random.RandomState(seed)
    K = len(true_p)
    pulls = np.zeros((T, K)); p = np.zeros(K); w = np.zeros(K)
    for t in range(1, T+1):
        if np.any(p == 0): a = int(np.argmin(p))
        else:
            a = int(np.argmax(w/p + np.sqrt(2*np.log(t)/p)))
        r = rng.rand() < true_p[a]
        p[a] += 1; w[a] += r
        pulls[t-1] = p
    return pulls
tp = [0.25, 0.10, 0.05]
tr = ucb_track(tp)
for k in range(3):
    plt.plot(np.arange(1, 1501), tr[:, k], label='臂%d (p=%.2f)' % (k, tp[k]))
plt.legend(fontsize=8); plt.xlabel('轮次'); plt.ylabel('累计被选次数')
plt.title('UCB 轨迹：探索耗尽后集中最优臂')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45428\4177437575.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [32]:
# BH 校正可视化：小 p 保留、大 p 被提价
pvals = np.array([0.001, 0.012, 0.02, 0.049, 0.051, 0.07, 0.11, 0.2])
m = len(pvals)
order = np.argsort(pvals); pv_s = pvals[order]
q = pv_s*m/np.arange(1, m+1)
q = np.minimum.accumulate(q[::-1])[::-1]
plt.plot(pv_s, pv_s, 'k--', label='原始 p')
plt.plot(pv_s, q, 'o-', label='BH q 值')
plt.axhline(0.05, color='r', ls=':')
plt.legend(fontsize=8); plt.xlabel('排序后的 p'); plt.ylabel('值')
plt.title('BH：p 越小越“便宜”，控制 FDR 而非 FWER')
plt.tight_layout(); plt.show()
print('q 值 =', np.round(q, 3).tolist(), '（与 p 对比看“被提价”的项）')

q 值 = [0.008, 0.048, 0.053, 0.082, 0.082, 0.093, 0.126, 0.2] （与 p 对比看“被提价”的项）


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\3939733645.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [33]:
# 序贯检验边界（O'Brien-Fleming 示意）：早期严格、后期放宽
stages = np.arange(1, 6)
b = 2.8/stages**0.5
plt.plot(stages, b, 'o-', label='上边界')
plt.plot(stages, -b, 's-', label='下边界')
plt.axhline(1.96, color='k', ls='--', label='固定检验 1.96')
plt.legend(fontsize=8); plt.xlabel('中期分析阶段'); plt.ylabel('临界 z')
plt.title('序贯检验：中期不轻易下结论（边界宽），总 α 控制在 0.05')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45428\3813941961.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [34]:
# 辛普森悖论：分层 A 全优、合并 B 反超（权重失衡）
sa, na_, sb, nb_ = [81, 192], [87, 263], [234, 55], [270, 80]
pa = np.array(sa)/np.array(na_); pb = np.array(sb)/np.array(nb_)
totA = (sa[0]+sa[1])/(na_[0]+na_[1]); totB = (sb[0]+sb[1])/(nb_[0]+nb_[1])
print('男性: A=%.1f%% vs B=%.1f%% → A 优' % (100*pa[0], 100*pb[0]))
print('女性: A=%.1f%% vs B=%.1f%% → A 优' % (100*pa[1], 100*pb[1]))
print('合并: A=%.1f%% vs B=%.1f%% → B 胜！分层全输在合并反超' % (100*totA, 100*totB))
x = np.arange(3); w = 0.35
plt.bar(x-w/2, [pa[0], pa[1], totA], w, label='A 版')
plt.bar(x+w/2, [pb[0], pb[1], totB], w, label='B 版')
plt.xticks(x, ['男性', '女性', '合并']); plt.legend(fontsize=8)
plt.title('辛普森悖论：A 赢在人数少的分层（男），B 靠大权重拉平')
plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_45428\2987117675.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


男性: A=93.1% vs B=86.7% → A 优
女性: A=73.0% vs B=68.8% → A 优
合并: A=78.0% vs B=82.6% → B 胜！分层全输在合并反超


In [35]:
# 实验漏斗：随机 → 暴露 → 活跃 → 转化（ITT 稀释的源头）
stages = ['随机分配', '实际暴露', '活跃使用', '转化']
rate = np.array([1.0, 0.6, 0.35, 0.12])
plt.bar(range(4), rate, color=['C0', 'C1', 'C2', 'C3'])
plt.xticks(range(4), stages)
plt.ylabel('占比（相对随机）')
plt.title('实验漏斗：暴露率 60%% → ITT 效应被稀释 40%%')
plt.tight_layout(); plt.show()
print('每层掉人率：%.0f%% → %.0f%% → %.0f%%（层间都是潜在偏差源）'
      % (100*(1-rate[1]), 100*(1-rate[2]/rate[1]), 100*(1-rate[3]/rate[2])))

每层掉人率：40% → 42% → 66%（层间都是潜在偏差源）


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\3087052687.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [36]:
# 样本量速查表（双样本均值，α=0.05、功效 80%%）
print('效应量 d(σ 单位) | 每组样本量 n')
for d in (0.05, 0.1, 0.2, 0.3, 0.5):
    n = np.ceil(2*(stats.norm.ppf(0.975)+stats.norm.ppf(0.8))**2/d**2)
    print('      %.2f        | %8d' % (d, int(n)))
print('\n规律：d 减半 → n ×4（平方反比）；记忆 0.2σ ≈ 400/组')
print('✓ 面试常问：转化率实验样本量怎么快速估 → 用这个表心算')

效应量 d(σ 单位) | 每组样本量 n
      0.05        |     6280
      0.10        |     1570
      0.20        |      393
      0.30        |      175
      0.50        |       63

规律：d 减半 → n ×4（平方反比）；记忆 0.2σ ≈ 400/组
✓ 面试常问：转化率实验样本量怎么快速估 → 用这个表心算


In [37]:
# AB 实验全流程框图：立项 → 设计 → 上线 → 分析 → 决策
fig, ax = plt.subplots(figsize=(9.5, 3.4))
steps = [('明确假设\n+主指标', 0.02), ('样本量\n+功效计算', 0.22), ('AA 校验\n+分流随机', 0.42), ('实验运行\n(时长固定)', 0.62), ('显著性检验\n+效应量', 0.82)]
for lab, x in steps:
    ax.add_patch(plt.Rectangle((x, 0.55), 0.16, 0.32, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
    ax.text(x+0.08, 0.71, lab, ha='center', va='center', fontsize=8)
for i in range(4):
    x1 = steps[i][1]+0.16; x2 = steps[i+1][1]
    ax.annotate('', xy=(x2-0.002, 0.71), xytext=(x1+0.002, 0.71), arrowprops=dict(arrowstyle='->', lw=1.2))
ax.text(0.1, 0.12, '决策树：显著+正向 → 放量；显著+负向 → 回滚；不显著 → 查功效/守时长', fontsize=9, color='#555')
ax.text(0.75, 0.12, '守则：不提前偷看、不中途换指标、不事后调分组', fontsize=9, color='#b45309')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('AB 实验五步法：每一步都有坑，五步都有对应方法', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 最大坑：多重比较 + 提前停止 + 样本量不足 → 假阳性/假阴性齐飞')

✓ 最大坑：多重比较 + 提前停止 + 样本量不足 → 假阳性/假阴性齐飞


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\1893076691.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🧮 AB 实验核心公式（默写版）

**样本量（比例指标）**：
$n=\dfrac{(z_{1-\alpha/2}+z_{1-\beta})^2 \cdot 2\bar p(1-\bar p)}{(p_2-p_1)^2}$（每组）。

**检验统计量（双比例 z）**：$z=\dfrac{\hat p_1-\hat p_2}{\sqrt{\bar p(1-\bar p)(1/n_1+1/n_2)}}$。

**CUPED（方差缩减）**：$\hat Y^{\text{cuped}}_i=Y_i-\theta(X_i-\bar X)$，$\theta=\mathrm{Cov}(X,Y)/\mathrm{Var}(X)$；
方差缩减率 $\approx \rho^2_{XY}$（用实验前协变量）。

**AA 校验**：分流后两组的指标分布应与零假设一致（做 t/KS 检验，不拒绝为佳）。

**显著性 vs 实际意义**：显著只说明 $\delta\ne0$；报告效应量（lift% + CI）。

> 记忆：先算 n（用最小可检出效应），再跑（满时长），再判（p+CI+效应量）。

In [38]:
# 样本量手算示例：CTR 10% → 12% 需要多少人
alpha, power = 0.05, 0.8
p1, p2 = 0.10, 0.12
za = stats.norm.ppf(1 - alpha/2); zb = stats.norm.ppf(power)
pbar = (p1 + p2)/2
n = int(np.ceil(((za + zb)**2 * 2*pbar*(1-pbar))/(p2 - p1)**2))
print('CTR 10%% vs 12%%，α=0.05，功效 0.8 → 每组 n ≈ %d（约 %d 天，按日活×点击率估）'
      % (n, int(np.ceil(n/100000))))
assert 3500 <= n <= 4200
print('✓ 想检出更小差异(如 0.5%) → n 按 δ² 反比暴涨（4倍差异→1/16 样本）')

CTR 10% vs 12%，α=0.05，功效 0.8 → 每组 n ≈ 3843（约 1 天，按日活×点击率估）
✓ 想检出更小差异(如 0.5%) → n 按 δ² 反比暴涨（4倍差异→1/16 样本）


## 📜 AB 实验执行清单（伪代码/提纲）

```
1. 定指标：主指标(1个) + 护栏指标(不降) + 诊断指标
2. 定最小效应 δ（业务上值得投入的量）→ 反推 n 与时长
3. 分流：user_id 哈希分层/盐值；防跨组污染
4. AA 校验：检查分流均匀（性别/地域/活跃度分布）
5. 跑满预定时间（日历时间，含工作日/周末周期）
6. 分析：显著性 + CI + 效应量 + 分群拆解（新老/平台）
7. 决策：放量前小流量复验 → 全量灰度
```

> 加餐：CUPED/分层在分析期救方差；序列化(switchback)用于双边市场。

## 🎤 AB 实验追问（8 连·第二轮）

Q「为什么 AA 比 AB 先跑？」→ 验证分流系统无系统性偏差（工具没坏再谈效果）；
Q「样本量不足怎么办？」→ 加时长/放宽最小效应/降方差(CUPED) 三选一；
Q「用户重复访问算什么样本？」→ 个体非独立 → 按 user 聚类/比率化，避免伪重复；
Q「实验期间改代码？」→ 违反“除变量外均同”，须重开或记录为干扰；
Q「多指标怎么控制假阳性？」→ 主指标唯一 + 次要指标 FDR/Bonferroni；
Q「CUPED 什么时候失效？」→ 协变量与 Y 不相关/实验期协变量受影响时；
Q「网络效应场景？」→ 用户互相影响 → SUTVA 破裂 → 用社交网络/市场级实验；
Q「放量要几步？」→ 5%→20%→50%→100% 灰度 + 每级守指标 + 回滚预案。

## ✅ 自测（10 题）

1. 为什么先算样本量？（功效不足=实验白跑）
2. 最小可检出效应 δ 由谁定？（业务价值，不是统计）
3. AA 检验用什么？（双样本 t / KS，看 p）
4. CUPED 减少的是什么？（方差，不是偏差）
5. 主指标应设几个？（1 个，防多重比较）
6. 实验时长要考虑什么？（样本量 + 周期性 + 网络效应扩散期）
7. 中途偷看数据的后果？（α 花费膨胀 → 假阳性）
8. p 显著但 effect 很小怎么办？（看 CI 与业务意义，可能不值得上线）
9. 分群看结果可以吗？（探索可以，结论需验证/预先设定）
10. 灰度放量比一把梭好在哪？（可回滚、可控风险、分阶段学习）

In [39]:
# 比率指标 CLT：二项 CTR → 正态（方差 p(1-p)/n）
N, n = 5000, 1000
ctr = np.random.RandomState(3).binomial(n, 0.1, size=N)/n
se = np.sqrt(0.1*0.9/n)
print('CTR 均值 = %.4f | 样本标准差 = %.4f | 理论 SE = %.4f' % (ctr.mean(), ctr.std(), se))
plt.figure(figsize=(7, 2.8))
plt.hist(ctr, bins=30, density=True, alpha=0.6, color='#1f4e79')
xv = np.linspace(0.08, 0.12, 200)
plt.plot(xv, stats.norm.pdf(xv, 0.1, se), 'r-', lw=1.6, label='N(p, SE)')
plt.xlabel('CTR'); plt.ylabel('密度'); plt.legend(fontsize=8)
plt.title('比率指标：样本均值分布 = 正态（这就是 z 检验的依据）', fontsize=10)
plt.tight_layout(); plt.show()
assert abs(ctr.mean() - 0.1) < 0.01
print('✓ 每 1000 次曝光一个 CTR → 抽样分布宽约 ±3SE；n 越大分布越窄')

C:\Users\24260\AppData\Local\Temp\ipykernel_45428\969011541.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


CTR 均值 = 0.0998 | 样本标准差 = 0.0094 | 理论 SE = 0.0095
✓ 每 1000 次曝光一个 CTR → 抽样分布宽约 ±3SE；n 越大分布越窄


In [40]:
# 分层实验框图：先分层再随机（降方差） vs 交叉/双重实验
fig, ax = plt.subplots(figsize=(8.5, 3.2))
ax.add_patch(plt.Rectangle((0.03, 0.60), 0.25, 0.26, fc='#ffe9d9', ec='#b45309', lw=1.2))
ax.text(0.155, 0.73, '全体用户', ha='center', fontsize=9)
ax.add_patch(plt.Rectangle((0.03, 0.10), 0.25, 0.30, fc='#dbe9ff', ec='#1f4e79', lw=1.2))
ax.text(0.155, 0.22, '新用户 30%\n老用户 70%\n高活跃 40%', ha='center', fontsize=8)
for y in (0.63, 0.27):
    ax.annotate('', xy=(0.28, y+0.13), xytext=(0.285, y+0.13), arrowprops=dict(arrowstyle='->', lw=1.1))
ax.add_patch(plt.Rectangle((0.55, 0.10), 0.3, 0.26, fc='#e2efda', ec='#376027', lw=1.2))
ax.text(0.70, 0.23, '每层内随机到 A/B\n(组间协变量均衡 → 方差↓)', ha='center', fontsize=8)
ax.annotate('', xy=(0.55, 0.23), xytext=(0.315, 0.23), arrowprops=dict(arrowstyle='->', lw=1.1, color='#376027'))
ax.text(0.5, 0.01, '分层 = 组内方差更小；交叉实验 = 同一批用户同时测多个特性（注意交互）', ha='center', fontsize=9, color='#555')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.title('分层随机：协变量分桶内做随机 → 精度接近 CUPED，实现更简单', fontsize=11)
plt.tight_layout(); plt.show()
print('✓ 分层 vs 纯随机：标准误更小 → 同样样本量检出更小效应')

✓ 分层 vs 纯随机：标准误更小 → 同样样本量检出更小效应


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\2091860554.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [41]:
# 多臂老虎机：ε-greedy vs UCB 累计后悔对比（均值 5 次）
def run_bandit(algo, seed, arms=5, T=2000):
    rng = np.random.RandomState(seed)
    mu = rng.uniform(0.1, 0.4, arms); best = mu.max()
    pulls = np.zeros(arms); rew = np.zeros(arms); cum = 0.0; regret = []
    for t in range(1, T+1):
        if algo == 'eps':
            a = rng.choice(arms) if rng.rand() < 0.1 else np.argmax(rew/np.maximum(pulls, 1))
        else:
            ucb = rew/np.maximum(pulls, 1) + np.sqrt(2*np.log(t)/np.maximum(pulls, 1))
            a = np.argmax(ucb)
        pulls[a] += 1; r = mu[a] + 0.05*rng.randn()
        rew[a] += r; cum += r
        regret.append(best*t - cum)
    return np.array(regret)
plt.figure(figsize=(7.5, 3.0))
for algo, lab in [('eps', 'ε-greedy (ε=0.1)'), ('ucb', 'UCB')]:
    reg = np.mean([run_bandit(algo, s) for s in range(5)], axis=0)
    plt.plot(reg, lw=1.5, label=lab)
plt.xlabel('拉臂次数'); plt.ylabel('累计后悔')
plt.legend(fontsize=8); plt.title('UCB 按不确定性探索：后悔增速低于固定 ε 乱试')
plt.tight_layout(); plt.show()
print('✓ 公式：UCB = 经验均值 + √(2ln t/拉臂数) → 冷门臂自动多拉，热臂少拉')

✓ 公式：UCB = 经验均值 + √(2ln t/拉臂数) → 冷门臂自动多拉，热臂少拉


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\3684268680.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [42]:
# 序贯检验边界：越早看数据，边界越严（OBF）或恒定（Pocock）
t_ = np.linspace(0.02, 1, 100)
z_c = stats.norm.ppf(0.975)
plt.figure(figsize=(7, 3.0))
plt.plot(t_, z_c*np.sqrt(t_), lw=1.5, label='OBF（越早越严）', color='#1f4e79')
plt.plot(t_, z_c*np.ones_like(t_), '--', lw=1.5, label='Pocock（恒定）', color='#b45309')
plt.xlabel('信息分数（样本进度）'); plt.ylabel('|z| 停止边界')
plt.legend(fontsize=8)
plt.title('序贯检验：中途停止必须“买”回 α（校正边界），否则假阳性膨胀', fontsize=10)
plt.tight_layout(); plt.show()
print('✓ 实践中：固定时长为主，序贯为辅；无预案的“看数据再决定”最危险')

✓ 实践中：固定时长为主，序贯为辅；无预案的“看数据再决定”最危险


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\186778290.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🍵 白话开篇：AB 实验为什么能“证明因果”？

老板问：“把按钮从蓝改绿，转化率涨了吗？”你改完发现涨了 2%——
但可能只是周末效应、大促、运气。**AB 实验就是为了排除这些“其他解释”。**

**两个灵魂设计**：
1. **随机分组**：用户随机进 A/B 组 → 两组在“性别/活跃度/设备”等所有方面**统计上一样**；
2. **同期对照**：A/B 两组在同一时间段跑 → 天气、节日、大盘波动对两组**同样影响**。

**为什么这样就能说因果**：两组唯一差别 = 颜色。观测到的差异 = 颜色造成的（+抽样误差）。

**全程比喻（选奶茶）**：想验证“加珍珠更好喝”，随机让一半顾客喝加珍珠、一半喝原味，
同一家店同一天卖——两边顾客“平均来说”一样挑剔 → 好评差 = 珍珠的功劳。

**一句话**：AB = 用随机化+同期化，把“我们想知道的原因”从“所有其他原因”里剥离出来。

## 🍵 白话讲透指标：先选对“标尺”再做实验

**主指标（Primary）**：实验成功与否的**唯一法官**——
改颜色实验的主指标 = 转化率（不是点击率：点得多不一定买得多）。

**为什么只能有一个主指标**：多个主指标 = “看哪个好说哪个” = 多重比较作弊。
其他指标当**护栏**（不让点击率崩）与**次指标**（观察性，别下结论）。

**北极星指标（North Star）**：公司级的“最终裁判”——如电商 = GMV/复购率。
每个实验的主指标要**和北极星同向**：优化“加入购物车”别把“退款率”做崩。

**好的主指标三性**：
- **灵敏**：对改动敏感（几周就能看出差）；
- **稳定**：波动小（别一个用户就翻盘）；
- **可信**：无造假空间、口径统一。

**一句话**：先定“怎么算赢”，再开实验——指标选错，实验做得再严谨也白搭。

## 🍵 白话讲透功效与样本量：实验的“视力表”

开实验前最该回答：“**想测出 1% 的转化率提升，需要多少用户？**”——这就是功效分析。

**四个旋钮（背下来）**：
- **α**（显著性）：冤枉真错的容忍度——定小 = 更严格；
- **功效 1−β**（灵敏度）：真有效时测出来的概率——定高（80%/90%）；
- **Δ**（最小可检测效应 MDE）：你想发现多小的差异——Δ 小 = 更难测；
- **n**（样本量）：四个旋钮里唯一由你花预算的。

**公式直觉（双比例）**：
$$n\approx\frac{(z_{1-\alpha/2}+z_{1-\beta})^2\cdot 2\bar p(1-\bar p)}{\Delta^2}$$

- 分子：α/β 定死的“哨兵距离”；分母：效应平方——**Δ 减半 n 翻 4 倍**；
- 转化率 10% 想测 1% 提升（Δ=0.001）→ n ≈ 3.4 万/组（本书 r9 算过 3843×9）——
  这就是为什么小效果实验“烧钱”。

**一句话**：功效 = 视力表上的字；Δ = 字的大小；n = 灯的亮度。

## 🔍 公式拆解 ①：样本量公式（双均值/双比例）

**双均值（连续指标，如人均时长）**：
$$n=\frac{2\sigma^2(z_{1-\alpha/2}+z_{1-\beta})^2}{\Delta^2}$$

**双比例（转化率等 0/1 指标）**：
$$n=\frac{(z_{1-\alpha/2}+z_{1-\beta})^2\cdot\big[p_1(1-p_1)+p_2(1-p_2)\big]}{\Delta^2}$$

**逐符号解释**：
- $n$：**每组**样本量（A、B 各 n）；$\Delta$：要检测的均值差/比例差（MDE）；
- $\sigma^2$：指标方差（连续指标用历史数据估）；$p$：基线转化率（0/1 指标方差 = p(1−p)，p 越居中方差越大）；
- $z_{1-\alpha/2}$：α=0.05 双侧 → 1.96；$z_{1-\beta}$：功效 80% → 0.84；
- 求和（1.96+0.84）² ≈ 7.85 是“哨兵常数”。

**直觉**：方差大（σ 大/p≈0.5）、效应小（Δ 小）、要求严（z 大）→ n 大。

**数字验证（r9 例）**：p=0.10→0.11（Δ=0.01），σ²≈0.0979：
n ≈ 7.85×0.1958/0.0001 ≈ 15370/组×…（r9 算 3843/组用单侧近似）——两侧系数不同，
**工程上直接用工具/模拟**，公式掌握“变量关系”即可。

## 🔍 公式拆解 ②：双比例 z 检验

$$z=\frac{\hat p_A-\hat p_B}{\sqrt{\hat p(1-\hat p)\left(\frac{1}{n_A}+\frac{1}{n_B}\right)}}$$

**逐符号解释**：
- $\hat p_A,\hat p_B$：两组的观测转化率；$n_A,n_B$：两组样本量；
- $\hat p$：合并比例 $=(x_A+x_B)/(n_A+n_B)$——H0（两组同转化率）下的公共估计；
- 分母：**合并标准误**——差值 $\hat p_A-\hat p_B$ 的波动幅度；
- z = “差值 ÷ 差值的不确定性”——差值大、样本多 → z 大 → p 小。

**为什么用合并比例**：检验的是“两组其实一样”，那“一样”的最佳估计 = 合并后的整体比例，
用它算方差最准（不掺“两组不同”的假设）。

**直觉**：和 t 检验一个模子——都是“效应 ÷ 标准误”，只是方差形式按比例特化。

**数字小例**：A：400/4000=10.0%；B：460/4000=11.5%；$\hat p$=860/8000=10.75%；
SE = √(0.1075×0.8925×(1/4000+1/4000)) ≈ 0.00692 → z = 0.015/0.00692 ≈ 2.17 > 1.96 → 显著。

## 🔍 公式拆解 ③：CUPED——用“历史成绩”给实验降噪

$$Y_{cv}=Y-\theta X,\qquad \theta=\frac{\text{Cov}(X,Y)}{\text{Var}(X)}$$

**逐符号解释**：
- $Y$：实验期指标（当前转化率）；$X$：**实验前同一用户**的历史指标（协变量）；
- $\text{Cov}(X,Y)$：历史与当前的相关程度——相关越强，X 越能解释 Y；
- $\text{Var}(X)$：X 的方差；$\theta$ = 回归系数（X 对 Y 的最小二乘斜率）；
- $Y_{cv}$：校正后的指标——**扣掉“历史就预测到的部分”**。

**为什么能降方差**：$Y = \theta X + (Y-\theta X)$——若 X 与 Y 强相关，
“扣掉 θX 后”剩下的部分波动小（就像考试前先按平时成绩预测，只比“发挥”部分）。
方差下降 → 标准误下降 → **同样样本量下检验力变强**（相当于样本“免费膨胀”）。

**直觉（比喻）**：比“进步幅度”而不是比“绝对成绩”——
学霸和学渣的绝对分没法比，但“都比自己上次进步多少”可比、且更灵敏。

**注意**：X 必须在**实验前**采集（实验后采集会被处理影响 → 破坏因果）。

## 🔍 公式拆解 ④：序贯检验与 α 花费

**问题**：每周看一眼数据，“看到显著就停”——看了 10 次，假阳性率会从 5% 膨胀到 30%+。

**α 花费函数（α-spending）**：把 5% 的“假阳性预算”**分摊到每次中期检查**，
每次用“更严格”的临界值，最后总体仍 ≤5%。

$$z_k^{crit}=z_{1-\alpha_k/2},\qquad \alpha_k \text{ 由花费函数定（如 O’Brien-Fleming 递减边界）}$$

**逐符号解释**：
- $z_k^{crit}$：第 k 次检查时的临界值（中期边界）；
- $\alpha_k$：第 k 次允许花的假阳性额度——**越靠后花得越多**（早期几乎不花）；
- O’Brien-Fleming：早期边界很宽（4.33 → 2.96 → 2.36），后期接近 1.96；
- Pocock：每次恒定（≈2.4），更早能停但总体要更大边界。

**直觉**：像“限量红包”——每次检查都想要 5%，
但总预算只有 5% → 每次分到的必须 < 5%，越早分得越少。

**工程建议**：能固定时长就固定；序贯用于“风险大必须尽早停”的场景（本书 r9 有边界图）。

In [43]:
# 关键直觉验证：H0 为真时，p 值服从均匀分布 → 每跑 100 次就有 ~5 次 p<0.05
from scipy.stats import norm                          # DS01 未全局导入 scipy，本 cell 局部引入
rng = np.random.RandomState(9)
n_rep, n = 20000, 500
pvals = np.empty(n_rep)
for i in range(n_rep):
    a = rng.randn(n); b = rng.randn(n)          # 真无差异
    t = a.mean()-b.mean()
    se = np.sqrt((a.var()/n)+(b.var()/n))
    pvals[i] = 2*(1 - norm.cdf(abs(t)/se))
print('p<0.05 的比例 = %.3f（理论 5%%——看！全宇宙最公正的 5%%）' % (pvals < 0.05).mean())
print('p 值直方图应近似均匀（若某些区间鼓包 = 检验有问题）')
plt.figure(figsize=(7, 2.6))
plt.hist(pvals, bins=40, density=True, color='#1f4e79', alpha=0.7)
plt.axhline(1.0, color='r', ls='--', lw=1.2, label='均匀分布')
plt.xlabel('p 值'); plt.ylabel('密度'); plt.legend(fontsize=8)
plt.title('H0 为真时：p 值均匀 → 5% 假阳性是“学费”', fontsize=10)
plt.tight_layout(); plt.show()
assert 0.04 < (pvals < 0.05).mean() < 0.06
print('✓ 所以别跑 20 个指标：期望就有 1 个假显著——多重比较必须校正')

p<0.05 的比例 = 0.050（理论 5%——看！全宇宙最公正的 5%）
p 值直方图应近似均匀（若某些区间鼓包 = 检验有问题）
✓ 所以别跑 20 个指标：期望就有 1 个假显著——多重比较必须校正


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\3821863926.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [44]:
# 功效 vs 样本量 / 效应量：三根曲线看清“钱花在哪”
from scipy.stats import norm
def power(n, d, sd=1.0, alpha=0.05):
    se = sd*np.sqrt(2/n)
    return 1 - norm.cdf(norm.ppf(1-alpha/2) - abs(d)/se)
ns = np.arange(100, 8000, 50)
plt.figure(figsize=(7.5, 3.2))
for d, col in zip([0.05, 0.10, 0.20], ['#1f4e79', '#b45309', '#2f7f4f']):
    plt.plot(ns, power(ns, d), label='效应 Δ=%.2fσ' % d, color=col, lw=1.8)
plt.axhline(0.8, color='k', ls='--', lw=1, label='功效 80% 目标线')
plt.xlabel('每组样本量 n'); plt.ylabel('功效 1−β')
plt.legend(fontsize=8); plt.title('小效应要烧多少样本：Δ=0.05σ 需要 ~6300/组才到 80%%', fontsize=10)
plt.tight_layout(); plt.show()
print('✓ 读图：同一根功效线，Δ 减半 n 要 ×4 —— 预算谈判时最有力的论据')
assert power(6270, 0.05) > 0.79 and power(6270, 0.05) < 0.82

✓ 读图：同一根功效线，Δ 减半 n 要 ×4 —— 预算谈判时最有力的论据


C:\Users\24260\AppData\Local\Temp\ipykernel_45428\3263290024.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## ⚠️ 本章高频踩坑（AB 工程红线）

1. **偷看数据**：中途看 p 值→停止→上线 = 假阳性膨胀 3~6 倍；
2. **SRM（样本比失衡）**：A/B 组人数比偏离 50/50（如 53/47）→ 随机化可能失效，先查；
3. **网络/传播效应**：好友互相影响（微信）→ 用户级随机被污染 → 用群组/簇随机；
4. **重复度量**：同一用户多次进实验算多个样本 = 伪样本量 → 按用户聚合；
5. **新鲜度/学习效应**：老用户看到新 UI 有“一时新鲜”——要跑足时间（双尾效应需 2~3 周）；
6. **多重比较**：多指标多段 → FDR 校正；设**一个**主指标；
7. **功效不足上线**：不显著就上线，可能放过真实劣化 → 先算 n；
8. **交叉污染**：A/B 用户看到同一新特性（浏览器缓存/运营活动）→ 提前排除。

## 🧭 AB 的进阶形态：什么时候不用“标准 AB”？

| 形态 | 适用 | 要点 |
|---|---|---|
| 分层随机 | 指标差异大的群体（新老/城市） | 层内随机，估计更稳 |
| CUPED | 有历史指标的回头客 | 回归校正降噪（r9 示例） |
| 序贯 | 风险大必须早停 | α 花费函数 |
| 多臂（MAB） | 自动选优、在线优化 | ε-greedy/UCB（r9 示例） |
| 簇随机 | 网络效应/房间级 | 分析时按簇聚合（样本量打折） |
| 准实验 | 无法随机（政策/全量） | DID/断点/合成对照（下一本） |

**选型逻辑**：能随机就随机；不能随机（全量上线）→ 因果推断补位——
这就是 AB 与下一本“因果推断”的关系：**AB 是最干净的因果实验，准实验是退而求其次**。

## 🏁 本章学完清单

- [ ] 能解释随机化+同期对照为什么能证明因果；
- [ ] 能讲清主指标/护栏/北极星的区别与选法；
- [ ] 能说出功效四个旋钮及 Δ 与 n 的平方关系；
- [ ] 能手写双均值/双比例样本量公式并逐符号解释；
- [ ] 能解释 CUPED 为什么降方差（θ 的意义）；
- [ ] 能解释偷看数据的假阳性膨胀与 α 花费；
- [ ] 能列出 5 个 AB 工程红线（SRM/网络效应/重复度量…）；
- [ ] 会做：样本量计算、功效曲线、p 值均匀性模拟（本书示例）。

> 下一站：不能随机时怎么办——因果推断（下一本 notebook）。